# Day 81 — Neural Networks

This notebook covers PyTorch DataLoaders (`torch.utils.data.DataLoader`), Mini-Batching (`batch_size`), Epoch Shuffling (`shuffle`), Remainder Batch Handling (`drop_last`), Worker Configuration (`num_workers`), and Dynamic Batch Size Hygiene

## Validation goals
- Understand the engineering benefits of subclassing `torch.nn.Module` including automatic parameter registration, device management, and state tracking
- Build a feedforward regression network (`DemandRegressorNet`) using `nn.Sequential`, `nn.Linear`, and `nn.ReLU`
- Distinguish between calling `model(x)` (`Module.__call__` with hooks) and invoking `model.forward(x)` directly
- Inspect layer shapes, parameter counts, and `requires_grad` flags via `model.named_parameters()`

### 1. Why `torch.nn.Module`?

In PyTorch, every neural network—and even custom building blocks—inherits from `torch.nn.Module`. Subclassing `nn.Module` gives you several critical engineering benefits out of the box:
- Automatic Parameter Registration: Any nn.Module attribute (such as` nn.Linear`) or nn.Parameter assigned inside __init__ is automatically tracked so optimizers can update it and `.to(device)` moves it cleanly.
- Structured Forward Pass: You define how data flows through your layers inside forward(`self, x`), keeping model logic modular and readable.
- State & Mode Management: Built-in helpers like `.parameters()`, `.state_dict()`, `.train()`, and `.eval()` make saving weights and toggling training/evaluation behavior straightforward.

### 2. Building a Feedforward Network with nn.Module

A standard feedforward neural network for tabular or structured data stacks nn.Linear layers with non-linear activations such as `nn.ReLU`:
- Call super().__init__() first inside __init__.
- Define layers (`nn.Linear`, `nn.ReLU`, etc.) as attributes in __init__.
- Connect them in forward(self, x) and return the output tensor.

For a regression target (like predicting continuous demand units), the final `nn.Linear(..., 1)` layer typically has no activation function so it can output unbounded real numbers. For binary classification, you might either apply `nn.Sigmoid()` or output raw logits and pair them with a logit-aware loss function.

In [1]:
import torch
import torch.nn as nn


class DemandRegressorNet(nn.Module):
    """Feedforward neural network for continuous tabular regression."""

    def __init__(self, in_features: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# Example forward pass with a batch of 8 samples, each having 12 features
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DemandRegressorNet(in_features=12, hidden_dim=64).to(device)

sample_batch = torch.randn((8, 12), dtype=torch.float32, device=device)
predictions = model(sample_batch)  # Calls forward() via nn.Module.__call__

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("--- Day 81: nn.Module Forward Pass ---")
print(f"Device:             {device}")
print(f"Input batch shape:  {sample_batch.shape}")
print(f"Output pred shape:  {predictions.shape}")
print(f"Trainable params:   {total_params}")

--- Day 81: nn.Module Forward Pass ---
Device:             cpu
Input batch shape:  torch.Size([8, 12])
Output pred shape:  torch.Size([8, 1])
Trainable params:   2945


Notice one subtle detail: we call model(`sample_batch`) rather than `model.forward`(`sample_batch`). Calling the instance invokes nn.Module.__call__, which runs registered hooks and then executes your forward method.

### 3. Inspecting Layers and Parameters

Understanding parameter counts helps you gauge model capacity and memory footprint:
- An `nn.Linear(in_features=12, out_features=64)` layer has a weight matrix of shape (`64, 12`) (`12 * 64 = 768 weights`) plus a bias vector of shape (`64,`) (`64 biases`), totaling 832 parameters.

You can inspect named parameters with for name, param in `model.named_parameters()`: to verify shapes and check that requires_grad=True on trainable weights.

In [2]:
for name, param in model.named_parameters():
    print(f"{name:20s} | shape: {str(tuple(param.shape)):12s} | requires_grad: {param.requires_grad}")

net.0.weight         | shape: (64, 12)     | requires_grad: True
net.0.bias           | shape: (64,)        | requires_grad: True
net.2.weight         | shape: (32, 64)     | requires_grad: True
net.2.bias           | shape: (32,)        | requires_grad: True
net.4.weight         | shape: (1, 32)      | requires_grad: True
net.4.bias           | shape: (1,)         | requires_grad: True


### Guided Questions:

Why do we call `model(x)` instead of `model.forward(x)` directly when running a PyTorch neural network?

### Answer:

In PyTorch, every neural network inherits from torch.nn.Module, which implements __call__. Calling `model(x)` triggers Module.__call__, whereas calling `model.forward(x)` invokes just your plain Python method directly.

**What `model(x)` does behind the scenes**

- Runs forward pre-hooks: Executes any hooks registered via register_forward_pre_hook (useful for inspecting or modifying inputs before the layer runs).
- Calls `self.forward(x)`: Computes the actual layer or model transformation.
- Runs forward hooks: Executes any hooks registered via register_forward_hook (commonly used to log activations, check shapes, or debug intermediate outputs).
- Attaches backward hooks: Wires up hooks needed for autograd and debugging gradients during backpropagation.

**Why `model.forward(x)` is a trap**

If you call `model.forward(x)` directly, none of those hooks fire. Any logging, gradient tracking hooks, profiling, or custom pre/post-processing attached to the module gets silently skipped—which can make debugging tricky when a hook you registered seems to do nothing.

(Note: Autograd itself tracks operations on tensors as forward executes, so loss.backward() still works if you call forward directly, though doing so breaks nn.Module hook conventions and is considered bad practice.)

### Guided Question:

For `nn.Linear(in_features=10, out_features=32)` with bias enabled, how many total trainable parameters (weights + biases) are created?

### Answer:

For `nn.Linear(in_features=10, out_features=32)` with bias enabled, 352 total trainable parameters are created:Weights: $10 \times 32 = 320$ parameters (shape (32, 10) in PyTorch)Biases: $32$ parameters (one per output feature, shape (32,))Total: $320 + 32 = 352$ parameters

### Guided Question:

In `DemandRegressorNet`, why does the final nn.Linear(hidden_dim // 2, 1) layer omit a ReLU activation at the end?

### Answer:

In a regression model like DemandRegressorNet, the final `nn.Linear(hidden_dim // 2, 1)` layer omits a `ReLU` activation so the network can output unconstrained continuous values across its full linear range rather than clamping everything to zero or above.

Why omitting `ReLU` on the output layer matters for `regressionNegative` values are blocked: `ReLU` computes $\max(0, x)$. If your target variable (or standardized/log-transformed/residual demand target) can be negative—or if a linear combination before the output layer happens to be `negative—ReLU` clips it to `0.0`.Zero-gradient "dead" outputs: Any prediction that lands below `0` gets flattened to `0` with a gradient of `0` during backpropagation, which can stall or distort learning near the lower end of your target range.Standard regression design: Hidden layers use non-linear activations like `ReLU` so the network can learn complex, curved relationships between input features. The final linear layer then projects those learned representations onto a continuous numeric scale (often paired with `MSELoss` or `SmoothL1Loss`).

**When you would add an output activation**

Even when predicting something physically non-negative like demand, people usually still leave the final layer raw (or predict `log-demand`) and let the loss function + data guide the scale. You would only add a final activation—such as Softplus (a smooth, non-zero-gradient positive mapping) or `Sigmoid—if` you have a hard domain requirement and want to avoid clipping gradients at zero.